# 🏛️ HIEROGLYPH NLP PIPELINE v8 — GRADUATION FINAL
### Gardiner Codes → TLA Deep Normalization → Hybrid RAG (Dense+BM25+RRF)
### → Detailed LLM Prompt (55 examples) → EN + AR + Sentiment + Intention (ML)
### → Full Evaluation Suite (BLEU, ROUGE, METEOR, chrF, Recall@K, MRR)
### → RAG vs LLM-Only Comparison → K-Value Optimization → Flask API

---
**Fixes & Upgrades over V7:**
- ✅ Intention classifier: ML-based (TF-IDF + cosine) — replaces weak keyword-only
- ✅ Prompt truncation fix: token-aware example selection (never exceeds 3500 tokens)
- ✅ Config paths: auto-detect Kaggle vs Colab vs local — no more hard-coded paths
- ✅ Arabic translation: verified via back-translation confidence check
- ✅ API error handling: structured error codes + retry logic + graceful degradation
- ✅ All V2 proven features preserved (10/10 ran fully)
- ✅ Duplicate removal, Train/Test split, RAG vs LLM-Only, Win/Loss, K-Opt


## CELL 0 — Install Dependencies

In [ ]:
# import subprocess, sys

# def pip(*pkgs):
#     result = subprocess.run(
#         [sys.executable, '-m', 'pip', 'install', '-q'] + list(pkgs),
#         capture_output=True, text=True
#     )
#     if result.returncode != 0:
#         print(result.stderr[-300:])

# pip('numpy==1.26.4')
# pip('protobuf==3.20.3')
# pip('transformers==4.44.2', 'sentencepiece', 'accelerate', 'safetensors==0.4.3', 'scipy')
# pip('flask', 'flask-cors', 'pyngrok')
# pip('faiss-cpu', 'sentence-transformers', 'datasets')
# pip('rapidfuzz')
# pip('rank-bm25')
# pip('rouge-score')
# pip('sacrebleu')
# pip('nltk')
# pip('scikit-learn')          # ✅ NEW: needed for ML intention classifier
# subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)
# print('✅ All dependencies installed')


## CELL 1 — Paths & Config (✅ Auto-detect: Kaggle / Colab / Local)

In [2]:
import os

# ── Auto-detect environment ──────────────────────────────────────
def _detect_env():
    if os.path.exists('/kaggle/input'):
        return 'kaggle'
    try:
        import google.colab  # noqa
        return 'colab'
    except ImportError:
        pass
    return 'local'

ENV = _detect_env()
print(f'🖥️  Environment detected: {ENV.upper()}')

# ── Dataset root ─────────────────────────────────────────────────
if ENV == 'kaggle':
    _DATA_ROOT = '/kaggle/input/datasets/mo3azkhaled/hieroglyph-data'
    _WORK_ROOT = '/kaggle/working'

elif ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    _DATA_ROOT = '/content/drive/MyDrive/hieroglyph_data'
    _WORK_ROOT = '/content'

else:
    _BASE_DIR = os.path.dirname(os.path.abspath(__file__))
    _DATA_ROOT = os.path.join(_BASE_DIR, 'data')
    _WORK_ROOT = os.path.join(_BASE_DIR, 'outputs')
    os.makedirs(_WORK_ROOT, exist_ok=True)

# ── Paths (UPDATED) ──────────────────────────────────────────────
GARDINER_PATH    = os.path.join(_DATA_ROOT, 'Update_gardiner_sign.csv')
DICT_WORD_PATH   = os.path.join(_DATA_ROOT, 'egyptian_dictionary.csv')
INTENTION_PATH   = os.path.join(_DATA_ROOT, 'intention_dataset.csv')

FAISS_INDEX_PATH = os.path.join(_WORK_ROOT, 'bbaw_faiss.index')
FAISS_META_PATH  = os.path.join(_WORK_ROOT, 'bbaw_meta.pkl')
BM25_CACHE_PATH  = os.path.join(_WORK_ROOT, 'bbaw_bm25.pkl')
TEST_SET_PATH    = os.path.join(_WORK_ROOT, 'tla_test_set.csv')

# ── Thresholds ───────────────────────────────────────────────────
SIMILARITY_THRESHOLD = 0.85
TOP_K_RAG            = 55
HYBRID_ALPHA         = 0.6
TRAIN_SPLIT          = 0.99
MAX_PROMPT_TOKENS    = 3500

# ── Check files ──────────────────────────────────────────────────
print("\n📂 Checking dataset files:\n")

for label, path in [
    ('Gardiner Sign List', GARDINER_PATH),
    ('Word Dictionary',    DICT_WORD_PATH),
    ('Intention Dataset',  INTENTION_PATH),
]:
    status = '✅ FOUND' if os.path.exists(path) else '❌ NOT FOUND'
    print(f'{status} → {label}')
    print(f'   {path}\n')

print(f'📁 Work dir : {_WORK_ROOT}')
print('✅ Config ready')

🖥️  Environment detected: KAGGLE

📂 Checking dataset files:

✅ FOUND → Gardiner Sign List
   /kaggle/input/datasets/mo3azkhaled/hieroglyph-data/Update_gardiner_sign.csv

✅ FOUND → Word Dictionary
   /kaggle/input/datasets/mo3azkhaled/hieroglyph-data/egyptian_dictionary.csv

✅ FOUND → Intention Dataset
   /kaggle/input/datasets/mo3azkhaled/hieroglyph-data/intention_dataset.csv

📁 Work dir : /kaggle/working
✅ Config ready


## CELL 2 — Gardiner Sign List

In [3]:
import pandas as pd

df_g       = pd.read_csv(GARDINER_PATH)
GARDINER_MAP = {}

for _, row in df_g.iterrows():
    code = str(row['code']).strip().lower()
    if not code or code == 'nan':
        continue
    GARDINER_MAP[code] = {
        'phonetic'     : str(row['phonetic']).strip()     if pd.notna(row.get('phonetic'))     else '',
        'phonetic_tla' : str(row['phonetic_tla']).strip() if pd.notna(row.get('phonetic_tla')) else '',
        'meaning'      : str(row['meaning']).strip()      if pd.notna(row.get('meaning'))       else '',
        'unicode'      : str(row['unicode']).strip()      if pd.notna(row.get('unicode'))       else '',
    }

print(f'✅ Gardiner Sign List loaded: {len(GARDINER_MAP)} signs')


✅ Gardiner Sign List loaded: 819 signs


## CELL 3 — Word Dictionary

In [4]:
df_word_dict = pd.read_csv(DICT_WORD_PATH)
WORD_DICT    = {}

for _, row in df_word_dict.iterrows():
    transliteration = str(row['transliteration']).strip().lower()
    english         = str(row['english']).strip() if pd.notna(row['english']) else ''
    if transliteration and transliteration != 'nan':
        WORD_DICT[transliteration] = english

print(f'✅ Word dictionary loaded: {len(WORD_DICT)} entries')


✅ Word dictionary loaded: 1274 entries


## CELL 4 — Intention Dataset + ML Classifier (✅ UPGRADED from keyword-only)

In [5]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df_intent    = pd.read_csv(INTENTION_PATH)
INTENTION_MAP = {}

# Build raw examples for TF-IDF training
_intent_labels   = []
_intent_examples = []

for _, row in df_intent.iterrows():
    intent_en = str(row['intention_en']).strip()
    intent_ar = str(row['intention_ar']).strip() if 'intention_ar' in df_intent.columns else ''
    keywords  = [kw.strip().lower() for kw in str(row['keywords']).split(',')]

    # Collect training text (keywords + any example sentences if column exists)
    example_text = ' '.join(keywords)
    if 'example_en' in df_intent.columns and pd.notna(row.get('example_en')):
        example_text += ' ' + str(row['example_en'])

    INTENTION_MAP[intent_en] = {
        'arabic'  : intent_ar,
        'keywords': set(keywords),
    }
    _intent_labels.append(intent_en)
    _intent_examples.append(example_text)

# ── Fit TF-IDF matrix once ───────────────────────────────────────
_tfidf_vec    = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)
_intent_matrix = _tfidf_vec.fit_transform(_intent_examples)

def get_intention_ml(text: str) -> dict:
    """
    ML-based intention classifier using TF-IDF + cosine similarity.
    Falls back to keyword counting when similarity is very low (< 0.05).
    ✅ Fixes: V7 keyword-only approach missed synonyms and paraphrases.
    """
    if not text or not text.strip():
        return {'intention_en': 'unknown', 'intention_ar': ''}

    # ML path
    query_vec = _tfidf_vec.transform([text.lower()])
    sims      = cosine_similarity(query_vec, _intent_matrix).flatten()
    best_idx  = int(np.argmax(sims))
    best_sim  = float(sims[best_idx])

    if best_sim >= 0.05:
        intent_en = _intent_labels[best_idx]
    else:
        # Keyword fallback for very short / unusual inputs
        tl, best, best_n = text.lower(), 'unknown', 0
        for intent, data in INTENTION_MAP.items():
            n = sum(1 for kw in data['keywords'] if kw.strip() in tl)
            if n > best_n:
                best_n, best = n, intent
        intent_en = best

    return {
        'intention_en'    : intent_en,
        'intention_ar'    : INTENTION_MAP.get(intent_en, {}).get('arabic', ''),
        'confidence'      : round(best_sim, 4),
    }

# ── Alias so rest of pipeline uses the upgraded function ─────────
get_intention = get_intention_ml

print(f'✅ Intention ML classifier ready: {len(INTENTION_MAP)} classes')
print(f'   TF-IDF vocab size: {len(_tfidf_vec.vocabulary_)}')

# Quick sanity check
_tests = [
    ('The king offers a sacrifice to the gods', 'ritual / offering'),
    ('He defeated the enemies in battle',        'military / war'),
    ('The sun rises over the horizon',           'nature / celestial'),
]
print('\n🧪 Classifier sanity checks:')
for txt, expected in _tests:
    res = get_intention_ml(txt)
    print(f'  "{txt[:40]}..."')
    print(f'  → {res["intention_en"]} (conf={res["confidence"]}) | expected≈{expected}')


✅ Intention ML classifier ready: 139 classes
   TF-IDF vocab size: 2311

🧪 Classifier sanity checks:
  "The king offers a sacrifice to the gods..."
  → offering (conf=0.1251) | expected≈ritual / offering
  "He defeated the enemies in battle..."
  → red (conf=0.1573) | expected≈military / war
  "The sun rises over the horizon..."
  → sunrise (conf=0.2092) | expected≈nature / celestial


## CELL 5 — TLA Deep Normalization (merged V2 + V7)

In [6]:
import re, unicodedata

EGYPTIAN_CHAR_MAP = {
    'ꜣ': 'a',  'ꞽ': 'i',  'y': 'y',  'ꜥ': 'a',  'w': 'w',
    'b': 'b',  'p': 'p',  'f': 'f',  'm': 'm',  'n': 'n',
    'r': 'r',  'h': 'h',  'ḥ': 'h',  'ḫ': 'kh', 'ẖ': 'kh',
    's': 's',  'š': 'sh', 'ḳ': 'q',  'q': 'q',  'k': 'k',
    'g': 'g',  't': 't',  'ṯ': 'tj', 'd': 'd',  'ḏ': 'dj',
    'ṭ': 't',  'ḍ': 'd',  'ṣ': 's',  'ẓ': 'z',
}

SUFFIXES_TO_REMOVE = ['=f', '=k', '=ṯ', '=s', '=sn', '=ꞽ', '=n', '=tn', '=fꞽ']

def normalize_tla(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ''
    text = re.sub(r'[()]', '', text)
    text = unicodedata.normalize('NFC', text)
    text = ''.join(c for c in text if not unicodedata.combining(c))
    text = text.lower()
    for egy, lat in EGYPTIAN_CHAR_MAP.items():
        text = text.replace(egy.lower(), lat)
    for suf in SUFFIXES_TO_REMOVE:
        pattern = re.escape(suf) + r'(?=[\s\.]|$)'
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    text = re.sub(r'\.+', '.', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip('. ')
    return text

_test_norm_sample = 'nḏ (w)di̯ r =s'
print(f'  Test: "{_test_norm_sample}" → "{normalize_tla(_test_norm_sample)}"')
print('✅ TLA normalization ready')


  Test: "nḏ (w)di̯ r =s" → "ndj wdi r"
✅ TLA normalization ready


## CELL 6 — Build FAISS + BM25 Indexes + Train/Test Split (✅ from V2)

In [7]:
import pickle, faiss, gc
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from rank_bm25 import BM25Okapi

EMBED_MODEL    = None
faiss_index    = None
bbaw_meta      = None
bm25_index     = None
df_test_global = None

if (os.path.exists(FAISS_INDEX_PATH) and
        os.path.exists(FAISS_META_PATH) and
        os.path.exists(BM25_CACHE_PATH)):
    print('Loading cached FAISS + BM25 indexes...')
    faiss_index = faiss.read_index(FAISS_INDEX_PATH)
    with open(FAISS_META_PATH, 'rb') as f:
        bbaw_meta = pickle.load(f)
    with open(BM25_CACHE_PATH, 'rb') as f:
        bm25_index = pickle.load(f)
    print(f'  FAISS vectors : {faiss_index.ntotal}')
    if os.path.exists(TEST_SET_PATH):
        df_test_global = pd.read_csv(TEST_SET_PATH)
        print(f'  Test set rows : {len(df_test_global)}')
    print('✅ Cached indexes loaded')
else:
    print('Building indexes from scratch...')
    dataset = load_dataset(
        'thesaurus-linguae-aegyptiae/tla-Earlier_Egyptian_original-v18-premium',
        split='train'
    )
    df_raw = pd.DataFrame(dataset)

    # ── Data cleaning ───────────────────────────────────────────
    cols_drop = [c for c in ['hieroglyphs', 'dateNotBefore', 'dateNotAfter'] if c in df_raw.columns]
    df_clean  = df_raw.drop(columns=cols_drop)
    df_clean  = df_clean.dropna(subset=['transliteration', 'translation'])
    df_clean  = df_clean[df_clean['transliteration'].str.strip() != '']
    df_clean  = df_clean[df_clean['translation'].str.strip() != '']
    df_clean  = df_clean.drop_duplicates(subset=['transliteration', 'translation'])  # ✅ V2 upgrade
    df_clean  = df_clean.sample(frac=1, random_state=42).reset_index(drop=True)
    print(f'  Clean records: {len(df_clean)}')

    # ── Train / Test split ──────────────────────────────────────
    split_idx      = int(len(df_clean) * TRAIN_SPLIT)
    df_train       = df_clean.iloc[:split_idx].copy()
    df_test_global = df_clean.iloc[split_idx:].copy()
    df_test_global.to_csv(TEST_SET_PATH, index=False)
    print(f'  Train: {len(df_train)} | Test: {len(df_test_global)}')

    # ── Embeddings ───────────────────────────────────────────────
    EMBED_MODEL = SentenceTransformer('BAAI/bge-m3', device='cpu')
    norms       = df_train['transliteration'].apply(normalize_tla).tolist()
    print('  Encoding embeddings...')
    vectors     = EMBED_MODEL.encode(norms, batch_size=128, show_progress_bar=True,
                                      normalize_embeddings=True).astype('float32')

    # ── FAISS ────────────────────────────────────────────────────
    dim         = vectors.shape[1]
    faiss_index = faiss.IndexFlatIP(dim)
    faiss_index.add(vectors)
    faiss.write_index(faiss_index, FAISS_INDEX_PATH)

    bbaw_meta   = {
        'transcriptions': df_train['transliteration'].tolist(),
        'translations'  : df_train['translation'].tolist(),
    }
    with open(FAISS_META_PATH, 'wb') as f:
        pickle.dump(bbaw_meta, f)

    # ── BM25 ─────────────────────────────────────────────────────
    tokenized  = [t.split() for t in norms]
    bm25_index = BM25Okapi(tokenized)
    with open(BM25_CACHE_PATH, 'wb') as f:
        pickle.dump(bm25_index, f)

    del vectors, df_train
    gc.collect()
    print(f'✅ Indexes built & cached | FAISS: {faiss_index.ntotal} vectors')


2026-04-05 03:47:04.475109: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775360824.724956     165 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775360824.793817     165 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775360825.342642     165 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775360825.342688     165 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775360825.342691     165 computation_placer.cc:177] computation placer alr

Building indexes from scratch...


README.md: 0.00B [00:00, ?B/s]

train.jsonl:   0%|          | 0.00/5.92M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

  Encoding embeddings...


Batches:   0%|          | 0/74 [00:00<?, ?it/s]

✅ Indexes built & cached | FAISS: 9408 vectors


## CELL 7 — Load Seed-X + Sentiment + spaCy

In [9]:
from transformers import AutoModelForCausalLM, PreTrainedTokenizerFast, pipeline
from safetensors import safe_open
import transformers.modeling_utils as _mu
import torch, gc, re, spacy

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

if 'seed_x_model' in globals():
    del globals()['seed_x_model']
gc.collect()
torch.cuda.empty_cache()

# ── Monkey-patch ──
_orig_safe_open = safe_open

class _PatchedSafeOpen:
    def __init__(self, path, framework, device='cpu'):
        self._f = _orig_safe_open(path, framework=framework, device=device)
    def metadata(self):
        m = self._f.metadata()
        return m if (m is not None and m.get('format')) else {'format': 'pt'}
    def keys(self):          return self._f.keys()
    def get_tensor(self, k): return self._f.get_tensor(k)
    def __enter__(self):     return self
    def __exit__(self, *a):  pass

import safetensors._safetensors_rust as _sr
_sr.safe_open = _PatchedSafeOpen
_mu.safe_open  = _PatchedSafeOpen

# ✅ التعديل الرئيسي: استخدام repo ID بدل path محلي غلط
MODEL_PATH = 'ByteDance-Seed/Seed-X-PPO-7B' if ENV == 'kaggle' else 'seed-x-ppo-7b'

seed_x_tokenizer = PreTrainedTokenizerFast.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
)
if seed_x_tokenizer.pad_token is None:
    seed_x_tokenizer.pad_token = seed_x_tokenizer.eos_token

seed_x_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
    device_map='auto',
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
seed_x_model.eval()

LANG_TAGS = {'English': 'en', 'German': 'de', 'Arabic': 'ar',
             'French': 'fr', 'Italian': 'it', 'Spanish': 'es'}

sentiment_pipe = pipeline('sentiment-analysis',
                           model='cardiffnlp/twitter-roberta-base-sentiment-latest',
                           device=-1, top_k=None)

nlp_spacy = spacy.load('en_core_web_sm')

print('✅ Models loaded')

Device: cuda


tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'LlamaTokenizer'. 
The class this function is called from is 'PreTrainedTokenizerFast'.
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/15.0G [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


✅ Models loaded


## CELL 8 — Core Helper Functions (✅ Arabic verification added)

In [10]:
from rapidfuzz import fuzz
import time

def assemble_tla(codes: list) -> str:
    tokens = []
    for c in codes:
        key  = c.strip().lower()
        info = GARDINER_MAP.get(key, {})
        ph   = info.get('phonetic_tla', '') or info.get('phonetic', '')
        tokens.append(ph if ph else key)
    return ' '.join(tokens)

def assemble_standard(codes: list) -> str:
    tokens = []
    for c in codes:
        key  = c.strip().lower()
        info = GARDINER_MAP.get(key, {})
        ph   = info.get('phonetic', '')
        tokens.append(ph if ph else key)
    return ' '.join(tokens)

def get_meanings(codes: list) -> list:
    return [GARDINER_MAP.get(c.strip().lower(), {}).get('meaning', '') for c in codes]

def get_embed_model():
    global EMBED_MODEL
    if EMBED_MODEL is None:
        EMBED_MODEL = SentenceTransformer('BAAI/bge-m3', device='cpu')
    return EMBED_MODEL

def fix_grammar(text: str) -> str:
    doc   = nlp_spacy(text)
    sents = [s.text.strip() for s in doc.sents]
    return ' '.join(s[0].upper() + s[1:] for s in sents if s)

def get_sentiment(text: str) -> dict:
    raw     = sentiment_pipe(text[:512])[0][0]
    label   = raw['label'].lower()
    mapping = {'positive': 'Positive', 'neutral': 'Neutral', 'negative': 'Negative'}
    return {'label': mapping.get(label, label), 'confidence': raw['score']}

def translate_seedx(text: str, src_lang: str, tgt_lang: str,
                    max_new_tokens: int = 150) -> str:
    tgt_tag = LANG_TAGS.get(tgt_lang, 'en')
    prompt  = f'Translate the following text into {tgt_lang}:\n{text}\n<{tgt_tag}>'
    inputs  = seed_x_tokenizer(prompt, return_tensors='pt',
                                truncation=True, max_length=512)
    inputs.pop('token_type_ids', None)
    dev    = next(seed_x_model.parameters()).device
    inputs = {k: v.to(dev) for k, v in inputs.items()}
    with torch.no_grad():
        out = seed_x_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=1, do_sample=False,
            no_repeat_ngram_size=3, repetition_penalty=1.5,
            pad_token_id=seed_x_tokenizer.pad_token_id,
            eos_token_id=seed_x_tokenizer.eos_token_id,
        )
    new_tok = out[0][inputs['input_ids'].shape[1]:]
    decoded = seed_x_tokenizer.decode(new_tok, skip_special_tokens=True).strip()
    parts   = re.split(r'[\n]|(?<=[.!?])\s', decoded)
    clean   = parts[0].strip() if parts else decoded
    return re.sub(r'[^\w\s\u0600-\u06FF.,!?\'\"\\-]', '', clean).strip()

# ── ✅ NEW: Arabic verification via back-translation ──────────────
def translate_arabic_verified(english_text: str, max_retries: int = 2) -> dict:
    """
    Translate English → Arabic, then back-translate Arabic → English
    to estimate translation confidence.
    Returns: {'arabic': str, 'verified': bool, 'back_translation': str, 'confidence': float}
    ✅ Fixes: V7 Arabic output was unverified — silent hallucinations possible.
    """
    arabic = translate_seedx(english_text, 'English', 'Arabic', max_new_tokens=200)

    # Back-translate for verification
    back_en = translate_seedx(arabic, 'Arabic', 'English', max_new_tokens=150)

    # Compute overlap between original English and back-translation
    orig_words = set(english_text.lower().split())
    back_words = set(back_en.lower().split())
    overlap    = len(orig_words & back_words) / max(len(orig_words), 1)

    verified = overlap >= 0.25  # at least 25% word overlap = likely correct

    return {
        'arabic'          : arabic,
        'verified'        : verified,
        'back_translation': back_en,
        'confidence'      : round(overlap, 3),
    }

print('✅ Core helpers ready (Arabic verification included)')


✅ Core helpers ready (Arabic verification included)


## CELL 9 — Hybrid Search (Dense FAISS + BM25 + RRF)

In [11]:
def _dense_search(query_norm: str, top_k: int) -> list:
    model = get_embed_model()
    q_vec = model.encode([query_norm], normalize_embeddings=True).astype('float32')
    if hasattr(faiss_index, 'nprobe'):
        faiss_index.nprobe = 10
    D, I = faiss_index.search(q_vec, top_k * 2)
    results = []
    for score, idx in zip(D[0], I[0]):
        if idx < 0:
            continue
        results.append({
            'idx'           : int(idx),
            'score'         : float(score),
            'transcription' : bbaw_meta['transcriptions'][idx],
            'translation'   : bbaw_meta['translations'][idx],
        })
    return results

def _bm25_search(query_norm: str, top_k: int) -> list:
    tokens      = query_norm.split()
    bm25_scores = bm25_index.get_scores(tokens)
    top_indices = np.argsort(bm25_scores)[::-1][:top_k * 2]
    results     = []
    for idx in top_indices:
        if bm25_scores[idx] > 0:
            results.append({
                'idx'           : int(idx),
                'score'         : float(bm25_scores[idx]),
                'transcription' : bbaw_meta['transcriptions'][idx],
                'translation'   : bbaw_meta['translations'][idx],
            })
    return results

def _rrf(dense: list, sparse: list, k: int = 60, alpha: float = HYBRID_ALPHA) -> list:
    scores = {}
    for rank, r in enumerate(dense):
        scores[r['idx']] = scores.get(r['idx'], 0) + alpha * (1 / (k + rank + 1))
    for rank, r in enumerate(sparse):
        scores[r['idx']] = scores.get(r['idx'], 0) + (1 - alpha) * (1 / (k + rank + 1))
    all_hits = {r['idx']: r for r in dense + sparse}
    return [
        {**all_hits[idx], 'rrf_score': sc, 'score': sc}
        for idx, sc in sorted(scores.items(), key=lambda x: -x[1])
    ]

def hybrid_search(query_raw: str, top_k: int = TOP_K_RAG) -> list:
    query_norm = normalize_tla(query_raw)
    dense  = _dense_search(query_norm, top_k)
    sparse = _bm25_search(query_norm, top_k)
    return _rrf(dense, sparse)[:top_k]

print('✅ Hybrid search ready')


✅ Hybrid search ready


## CELL 10 — Token-Aware RAG Prompt Builder (✅ FIXED truncation bug)

In [12]:
def _count_tokens(text: str) -> int:
    """Fast approximate token count (chars/4 ≈ GPT tokens)."""
    return len(text) // 4

def _build_rag_prompt(query_original, query_norm, retrieved, meanings):
    """
    Build RAG prompt with strict token budget enforcement.
    ✅ Fixes V7 bug: V7 blindly included all 55 examples regardless of size,
    causing prompts to exceed model context and get silently truncated.
    Now we add examples one-by-one until MAX_PROMPT_TOKENS is reached.
    """
    meaning_hint = ', '.join(m for m in meanings if m)

    header = f"""You are a senior linguist specialising in Earlier Egyptian (Old Egyptian & Early Middle Egyptian),
with deep expertise in morphology, syntax, and historical semantics.

Your task: translate the Earlier Egyptian transliteration below into **German**,
using the retrieved corpus examples ONLY as structural and semantic guidance.

══════════════════════════════
INPUT TRANSLITERATION : {query_original}
NORMALIZED FORM       : {query_norm}
SIGN MEANINGS         : {meaning_hint or 'N/A'}
══════════════════════════════

RETRIEVED CORPUS EXAMPLES (most similar first):
"""

    footer = """
══════════════════════════════
STRICT RULES:
- Output ONLY the German translation.
- Do NOT copy example translations verbatim.
- Preserve morphological detail (case, tense, voice).
- Do NOT add explanations or alternative readings.

German Translation:"""

    budget_used = _count_tokens(header) + _count_tokens(footer)
    examples_text = ''
    included = 0

    for i, ex in enumerate(retrieved, 1):
        block = f"""
Example {i}:
- Original transcription : {ex['transcription']}
- German translation     : {ex['translation']}
- RRF score              : {ex.get('rrf_score', ex.get('score', 0)):.4f}
---"""
        block_tokens = _count_tokens(block)
        if budget_used + block_tokens > MAX_PROMPT_TOKENS:
            break
        examples_text += block
        budget_used   += block_tokens
        included       += 1

    print(f'  [Prompt] {included}/{len(retrieved)} examples used | ~{budget_used} tokens')
    return header + examples_text + footer


def translate_with_rag_prompt(query_original, query_norm, retrieved, meanings) -> str:
    prompt  = _build_rag_prompt(query_original, query_norm, retrieved, meanings)
    inputs  = seed_x_tokenizer(prompt, return_tensors='pt',
                                truncation=True, max_length=4096)
    inputs.pop('token_type_ids', None)
    dev    = next(seed_x_model.parameters()).device
    inputs = {k: v.to(dev) for k, v in inputs.items()}
    with torch.no_grad():
        out = seed_x_model.generate(
            **inputs,
            max_new_tokens=200,
            num_beams=2, do_sample=False,
            no_repeat_ngram_size=3, repetition_penalty=1.4,
            pad_token_id=seed_x_tokenizer.pad_token_id,
            eos_token_id=seed_x_tokenizer.eos_token_id,
        )
    new_tok = out[0][inputs['input_ids'].shape[1]:]
    return seed_x_tokenizer.decode(new_tok, skip_special_tokens=True).strip()

print('✅ Token-aware RAG prompt builder ready')


✅ Token-aware RAG prompt builder ready


## CELL 11 — Path Handlers (sentence / word / fallback)

In [13]:
def sentence_path(codes, rag_results, meanings):
    query_orig  = assemble_tla(codes)
    query_norm  = normalize_tla(query_orig)
    german      = translate_with_rag_prompt(query_orig, query_norm, rag_results, meanings)
    english     = fix_grammar(translate_seedx(german, 'German', 'English'))
    if not english or set(english.strip()) <= set('. -'):
        meaning_hint = ', '.join(m for m in meanings if m)
        english = meaning_hint if meaning_hint else query_orig
    sentiment = get_sentiment(english)
    intention = get_intention(english)                          # ✅ now uses ML classifier
    ar_result = translate_arabic_verified(english)             # ✅ verified Arabic
    arabic    = ar_result['arabic']
    best      = rag_results[0]
    return {
        'path': 'sentence', 'english': english, 'arabic': arabic,
        'arabic_verified'  : ar_result['verified'],
        'arabic_confidence': ar_result['confidence'],
        'sentiment': sentiment, 'intention': intention,
        'rag_score': best['score'], 'rag_source': best['transcription'],
        'rag_results': rag_results,
    }


def word_path(codes):
    std_phonetic   = assemble_standard(codes)
    tokens         = std_phonetic.lower().split()
    meanings_found = []
    for token in tokens:
        hit = WORD_DICT.get(token)
        if hit:
            meanings_found.append(hit)
    if not meanings_found:
        for token in tokens:
            for key, val in WORD_DICT.items():
                if fuzz.ratio(token, key) >= 80:
                    meanings_found.append(val)
                    break
    english   = ' / '.join(meanings_found) if meanings_found else std_phonetic
    ar_result = translate_arabic_verified(english)
    arabic    = ar_result['arabic']
    return {
        'path': 'word', 'phonetic': std_phonetic,
        'english': english, 'arabic': arabic,
        'arabic_verified'  : ar_result['verified'],
        'arabic_confidence': ar_result['confidence'],
        'sentiment': get_sentiment(english), 'intention': get_intention(english),
        'rag_score': 1.0, 'rag_source': '', 'rag_results': [],
    }

print('✅ Path handlers ready')


✅ Path handlers ready


## CELL 12 — Full Pipeline

In [14]:
def full_pipeline(gardiner_input: str, verbose: bool = True) -> dict:
    t0           = time.time()
    codes        = gardiner_input.strip().upper().split()
    tla_query    = assemble_tla(codes)
    tla_norm     = normalize_tla(tla_query)
    tla_combined = tla_norm.replace(' ', '')
    meanings     = get_meanings(codes)

    if verbose:
        print(f'Input codes  : {codes}')
        print(f'TLA raw      : {tla_query}')
        print(f'TLA norm     : {tla_norm}')

    direct_match = WORD_DICT.get(tla_combined)
    if not direct_match:
        for key, val in WORD_DICT.items():
            if fuzz.ratio(tla_combined, key.lower()) >= 85:
                direct_match = val
                break

    if direct_match:
        english   = direct_match.split('/')[0].strip()
        ar_result = translate_arabic_verified(english)
        arabic    = ar_result['arabic']
        if verbose:
            print(f'Path: WORD — dict hit: {english}')
        result = {
            'path': 'word', 'phonetic': tla_combined,
            'english': english, 'arabic': arabic,
            'arabic_verified'  : ar_result['verified'],
            'arabic_confidence': ar_result['confidence'],
            'sentiment': get_sentiment(english), 'intention': get_intention(english),
            'rag_score': 1.0, 'rag_source': '', 'rag_results': [],
        }
    else:
        rag_results = hybrid_search(tla_query, top_k=TOP_K_RAG)
        best_score  = rag_results[0]['score'] if rag_results else 0.0
        if verbose:
            print(f'Hybrid RAG best RRF score: {best_score:.4f}')

        if best_score >= SIMILARITY_THRESHOLD:
            if verbose: print('Path: SENTENCE')
            result = sentence_path(codes, rag_results, meanings)
        else:
            if verbose: print('Path: FALLBACK')
            word_meanings = [m for m in meanings if m]
            meaning_hint  = ', '.join(word_meanings)
            best_german   = ''
            for r in rag_results:
                if len(r['translation'].replace('.', '').replace(' ', '')) > 10:
                    best_german = r['translation']
                    break
            if best_german:
                ctx     = f'{best_german} (context: {meaning_hint})'
                english = fix_grammar(translate_seedx(ctx, 'German', 'English'))
            elif word_meanings:
                ctx     = f'Ancient Egyptian signs meaning: {meaning_hint}. Translate as a coherent sentence.'
                english = fix_grammar(translate_seedx(ctx, 'English', 'English'))
            else:
                english = tla_query
            if not english or set(english.strip()) <= set('. -'):
                english = meaning_hint if word_meanings else tla_query
            ar_result = translate_arabic_verified(english)
            arabic    = ar_result['arabic']
            result = {
                'path': 'fallback', 'english': english, 'arabic': arabic,
                'arabic_verified'  : ar_result['verified'],
                'arabic_confidence': ar_result['confidence'],
                'sentiment': get_sentiment(english), 'intention': get_intention(english),
                'rag_score': best_score,
                'rag_source': rag_results[0]['transcription'] if rag_results else '',
                'rag_results': rag_results,
            }

    result.update({
        'input': gardiner_input, 'tla_phonemes': tla_query,
        'tla_normalised': tla_norm, 'meanings': meanings,
        'processing_time': round(time.time() - t0, 2),
    })

    if verbose:
        print('=' * 60)
        print(f'PATH          : {result["path"].upper()}')
        print(f'TLA norm      : {tla_norm}')
        print(f'English       : {result["english"]}')
        print(f'Arabic        : {result["arabic"]}')
        ar_ver = result.get('arabic_verified')
        if ar_ver is not None:
            print(f'Arabic check  : {"✅ verified" if ar_ver else "⚠️  low confidence"} ({result.get("arabic_confidence",0):.0%})')
        s = result.get('sentiment')
        if s: print(f'Sentiment     : {s["label"]} ({s["confidence"]:.0%})')
        i = result.get('intention')
        if i: print(f'Intention     : {i["intention_en"]} / {i["intention_ar"]}')
        print(f'RAG Score     : {result["rag_score"]:.4f}')
        print(f'Time          : {result["processing_time"]}s')
        print('=' * 60)
    return result

# Smoke test
result = full_pipeline('G17 N35 D21')


Input codes  : ['G17', 'N35', 'D21']
TLA raw      : m n r
TLA norm     : m n r
Path: WORD — dict hit: monument
PATH          : WORD
TLA norm      : m n r
English       : monument
Arabic        : نصب تذكاري.
Arabic check  : ⚠️  low confidence (0%)
Sentiment     : Neutral (66%)
Intention     : fame / الشهرة والمجد
RAG Score     : 1.0000
Time          : 23.78s


## CELL 13 — Full Evaluation Suite (Translation + Retrieval)

In [15]:
import nltk
from nltk.translate.bleu_score   import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score                 import rouge_scorer
from sacrebleu.metrics           import CHRF
import warnings; warnings.filterwarnings('ignore')

nltk.download('wordnet', quiet=True)
nltk.download('punkt',   quiet=True)
nltk.download('omw-1.4', quiet=True)

def calc_bleu(reference, hypothesis):
    ref_tok = reference.lower().split()
    hyp_tok = hypothesis.lower().split()
    if not ref_tok or not hyp_tok: return 0.0
    smoother = SmoothingFunction().method1
    max_n    = min(len(ref_tok), len(hyp_tok), 4)
    weights  = {1:(1,0,0,0), 2:(.5,.5,0,0), 3:(.33,.33,.34,0)}.get(max_n, (.25,.25,.25,.25))
    return sentence_bleu([ref_tok], hyp_tok, weights=weights, smoothing_function=smoother) * 100

def calc_rouge(reference, hypothesis):
    scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
    scores = scorer.score(reference.lower(), hypothesis.lower())
    return {k: v.fmeasure * 100 for k, v in scores.items()}

def calc_meteor(reference, hypothesis):
    try:
        return meteor_score([reference.lower().split()], hypothesis.lower().split()) * 100
    except: return 0.0

def calc_chrf(reference, hypothesis):
    return CHRF().sentence_score(hypothesis, [reference]).score

def calc_exact_match(reference, hypothesis):
    return 100.0 if reference.lower().strip() == hypothesis.lower().strip() else 0.0

def calc_word_overlap(reference, hypothesis):
    ref_words = set(reference.lower().split())
    hyp_words = set(hypothesis.lower().split())
    if not ref_words: return 0.0
    return len(ref_words & hyp_words) / len(ref_words) * 100

def calc_recall_at_k(reference_german, retrieved, k_values=[1,3,5,10,20]):
    ref_norm = normalize_tla(reference_german).lower()
    scores   = {}
    for k in k_values:
        hit = any(
            fuzz.ratio(normalize_tla(r['translation']).lower(), ref_norm) >= 70
            for r in retrieved[:k]
        )
        scores[f'recall@{k}'] = 100.0 if hit else 0.0
    return scores

def calc_mrr(reference_german, retrieved):
    ref_norm = normalize_tla(reference_german).lower()
    for rank, r in enumerate(retrieved, 1):
        if fuzz.ratio(normalize_tla(r['translation']).lower(), ref_norm) >= 70:
            return 100.0 / rank
    return 0.0

def evaluate_pipeline(gardiner_input, reference_english='', reference_german=''):
    result = full_pipeline(gardiner_input, verbose=False)
    hyp_en = result.get('english', '')
    rouge  = calc_rouge(reference_english, hyp_en) if reference_english else {}
    recall = calc_recall_at_k(reference_german, result.get('rag_results', [])) if reference_german else {}
    mrr    = calc_mrr(reference_german, result.get('rag_results', [])) if reference_german else 0.0
    avg_r  = np.mean(list(recall.values())) if recall else 0.0

    return {
        **result,
        'reference_english'  : reference_english,
        'bleu'               : calc_bleu(reference_english, hyp_en)       if reference_english else 0.0,
        'rouge1'             : rouge.get('rouge1', 0.0),
        'rouge2'             : rouge.get('rouge2', 0.0),
        'rougeL'             : rouge.get('rougeL', 0.0),
        'meteor'             : calc_meteor(reference_english, hyp_en)      if reference_english else 0.0,
        'chrf'               : calc_chrf(reference_english, hyp_en)        if reference_english else 0.0,
        'exact_match'        : calc_exact_match(reference_english, hyp_en) if reference_english else 0.0,
        'word_overlap'       : calc_word_overlap(reference_english, hyp_en)if reference_english else 0.0,
        **recall,
        'mrr'                : mrr,
        'avg_retrieval_score': avg_r,
    }

print('✅ Evaluation suite ready')


✅ Evaluation suite ready


## CELL 14 — ✅ RAG vs LLM-Only Full Comparison (from V2)

In [16]:
def translate_llm_only(query_original: str) -> str:
    """Direct Seed-X translation WITHOUT any RAG retrieval context."""
    prompt = f"""You are an expert linguist specialized in Earlier Egyptian grammar and historical translation.

Translate the following Earlier Egyptian transliteration into German.
Use a conservative, grammar-based interpretation.
Do not modernize meanings or add implied words.

Egyptian Transliteration:
{query_original}

Output ONLY the German translation. Do not add explanations or alternative readings.

German Translation:
"""
    inputs = seed_x_tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512)
    inputs.pop('token_type_ids', None)
    dev    = next(seed_x_model.parameters()).device
    inputs = {k: v.to(dev) for k, v in inputs.items()}
    with torch.no_grad():
        out = seed_x_model.generate(
            **inputs, max_new_tokens=150,
            num_beams=1, do_sample=False,
            no_repeat_ngram_size=3, repetition_penalty=1.5,
            pad_token_id=seed_x_tokenizer.pad_token_id,
            eos_token_id=seed_x_tokenizer.eos_token_id,
        )
    new_tok = out[0][inputs['input_ids'].shape[1]:]
    return seed_x_tokenizer.decode(new_tok, skip_special_tokens=True).strip()


def run_full_comparison(num_samples: int = None):
    """Run RAG vs LLM-Only comparison on the test set."""
    if df_test_global is None or len(df_test_global) == 0:
        print('❌ Test set not available. Run CELL 6 first.')
        return None, None

    test_df = df_test_global if num_samples is None else df_test_global.head(num_samples)
    print('=' * 70)
    print(f'🆚 RAG vs LLM-ONLY COMPARISON ({len(test_df)} samples)')
    print('=' * 70)

    rag_results_list = []
    llm_results_list = []

    for idx in range(len(test_df)):
        row           = test_df.iloc[idx]
        query_orig    = row['transliteration']
        reference_de  = row['translation']
        reference_en  = translate_seedx(reference_de, 'German', 'English')

        # RAG path
        rag = evaluate_pipeline(query_orig, reference_english=reference_en,
                                reference_german=reference_de)
        rag_results_list.append(rag)

        # LLM-Only path
        llm_german = translate_llm_only(query_orig)
        llm_english = translate_seedx(llm_german, 'German', 'English')
        rouge_llm   = calc_rouge(reference_en, llm_english)
        llm_results_list.append({
            'sample_id'         : idx,
            'bleu'              : calc_bleu(reference_en, llm_english),
            'rouge1'            : rouge_llm.get('rouge1', 0.0),
            'rouge2'            : rouge_llm.get('rouge2', 0.0),
            'rougeL'            : rouge_llm.get('rougeL', 0.0),
            'meteor'            : calc_meteor(reference_en, llm_english),
            'chrf'              : calc_chrf(reference_en, llm_english),
            'exact_match'       : calc_exact_match(reference_en, llm_english),
            'word_overlap'      : calc_word_overlap(reference_en, llm_english),
            'recall@1': 0.0, 'recall@3': 0.0, 'recall@5': 0.0,
            'recall@10': 0.0, 'recall@20': 0.0, 'mrr': 0.0,
        })

        if (idx + 1) % 10 == 0:
            print(f'  Processed {idx + 1}/{len(test_df)} samples...')

    df_rag = pd.DataFrame(rag_results_list)
    df_llm = pd.DataFrame(llm_results_list)

    metric_cols = ['bleu','rouge1','rouge2','rougeL','meteor','chrf',
                   'exact_match','word_overlap','recall@1','recall@3',
                   'recall@5','recall@10','recall@20','mrr']

    print('\n📊 COMPARISON SUMMARY')
    print(f'{"Metric":<20} {"RAG":>10} {"LLM-Only":>10} {"Δ":>10}')
    print('-' * 55)
    for m in metric_cols:
        rag_mean = df_rag[m].mean() if m in df_rag.columns else 0.0
        llm_mean = df_llm[m].mean() if m in df_llm.columns else 0.0
        delta    = rag_mean - llm_mean
        bar      = '▲' if delta > 0 else ('▼' if delta < 0 else '─')
        print(f'{m:<20} {rag_mean:>10.2f} {llm_mean:>10.2f} {bar}{abs(delta):>8.2f}')

    # Win / Loss
    wins = {'RAG': 0, 'LLM': 0, 'Tie': 0}
    for m in ['bleu', 'meteor', 'chrf']:
        for i in range(min(len(df_rag), len(df_llm))):
            r = df_rag.iloc[i][m] if m in df_rag.columns else 0
            l = df_llm.iloc[i][m] if m in df_llm.columns else 0
            if r > l:   wins['RAG'] += 1
            elif l > r: wins['LLM'] += 1
            else:       wins['Tie'] += 1
    total = sum(wins.values())
    print(f'\n🏆 Win/Loss: RAG {wins["RAG"]/total:.0%} | LLM {wins["LLM"]/total:.0%} | Tie {wins["Tie"]/total:.0%}')

    df_rag.to_csv(os.path.join(_WORK_ROOT, 'rag_eval_results.csv'), index=False)
    df_llm.to_csv(os.path.join(_WORK_ROOT, 'llm_eval_results.csv'), index=False)
    print('\n💾 Results saved to CSV files')
    return df_rag, df_llm

# Run comparison (uncomment when ready)
# df_rag_results, df_llm_results = run_full_comparison(num_samples=50)


## CELL 15 — ✅ K-Value Optimization (from V2)

In [17]:
def run_k_optimization(num_samples: int = 10,
                        k_min: int = 5, k_max: int = 205, k_step: int = 5):
    """
    Test different TOP_K values on num_samples test sentences.
    Finds the optimal K for retrieval quality.
    Saves results to CSV.
    """
    global TOP_K_RAG

    if df_test_global is None or len(df_test_global) == 0:
        print('❌ Test set not available. Run CELL 6 first.')
        return None

    k_values = list(range(k_min, k_max, k_step))
    test_df  = df_test_global.head(num_samples)

    print('=' * 70)
    print(f'🔬 K-VALUE OPTIMIZATION ({num_samples} samples, K={k_min}..{k_max-1})')
    print('=' * 70)

    # Pre-compute queries and reference embeddings once
    test_samples = []
    for idx in range(len(test_df)):
        q_orig  = test_df.iloc[idx]['transliteration']
        ref_de  = test_df.iloc[idx]['translation']
        ref_en  = translate_seedx(ref_de, 'German', 'English')
        q_norm  = normalize_tla(q_orig)
        test_samples.append({'q_orig': q_orig, 'q_norm': q_norm,
                              'ref_de': ref_de, 'ref_en': ref_en})

    k_results = []
    for k in k_values:
        TOP_K_RAG = k
        bleu_scores, recall_scores = [], []
        for s in test_samples:
            rag     = hybrid_search(s['q_orig'], top_k=k)
            recall  = calc_recall_at_k(s['ref_de'], rag, k_values=[k])
            en_out  = translate_seedx(
                translate_with_rag_prompt(s['q_orig'], s['q_norm'], rag, []),
                'German', 'English'
            ) if rag else ''
            bleu_scores.append(calc_bleu(s['ref_en'], en_out))
            recall_scores.append(recall.get(f'recall@{k}', 0.0))

        avg_bleu   = np.mean(bleu_scores)
        avg_recall = np.mean(recall_scores)
        composite  = 0.5 * avg_bleu + 0.5 * avg_recall
        k_results.append({'k': k, 'bleu': avg_bleu,
                          f'recall@{k}': avg_recall, 'composite': composite})
        print(f'  K={k:3d} | BLEU={avg_bleu:5.2f} | Recall={avg_recall:5.2f} | Composite={composite:5.2f}')

    df_k = pd.DataFrame(k_results)
    best_row  = df_k.loc[df_k['composite'].idxmax()]
    TOP_K_RAG = int(best_row['k'])
    print(f'\n🏆 Optimal K = {TOP_K_RAG} (composite score = {best_row["composite"]:.2f})')
    df_k.to_csv(os.path.join(_WORK_ROOT, 'k_optimization_results.csv'), index=False)
    print(f'💾 Saved to k_optimization_results.csv')
    return df_k

# Run optimization (uncomment when ready)
# df_k_results = run_k_optimization(num_samples=10)


## CELL 16 — Kill Old Flask Process

In [ ]:
import os
os.system('kill -9 $(lsof -t -i:5006) 2>/dev/null')
time.sleep(2)
print('✅ Port 5006 cleared')


## CELL 17 — Flask API + Ngrok (✅ Structured error handling)

In [20]:
import threading
from pyngrok import ngrok, conf
from flask import Flask, request, jsonify
from flask_cors import CORS

ngrok.kill(); time.sleep(2)

NGROK_TOKEN = "3BBILR8WLOgh6uQGi4jU0aN6YoR_68QjQ4J4eRxCyhsjHLWMr"   # ← replace with your token
conf.get_default().auth_token = NGROK_TOKEN

app = Flask(__name__)
CORS(app)

# ── Structured error codes ────────────────────────────────────────
# ✅ Fixes V7: bare except → str(e) gave no context.
# Now every error returns code + message + hint.

ERROR_CODES = {
    'MISSING_INPUT' : ('400', 'Required field is missing'),
    'INVALID_INPUT' : ('400', 'Input is malformed or empty'),
    'PIPELINE_ERROR': ('500', 'Internal pipeline failure'),
    'MODEL_ERROR'   : ('503', 'Language model unavailable'),
}

def _api_error(code_key: str, detail: str = '', status: int = None):
    code, msg = ERROR_CODES.get(code_key, ('500', 'Unknown error'))
    return jsonify({
        'success': False,
        'error'  : {'code': code_key, 'message': msg, 'detail': detail},
    }), status or int(code)

def _run_pipeline_safe(gardiner_str: str):
    """Wrapper with retry on transient failures."""
    last_exc = None
    for attempt in range(2):
        try:
            return full_pipeline(gardiner_str, verbose=False), None
        except RuntimeError as e:
            last_exc = e
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        except Exception as e:
            return None, str(e)
    return None, str(last_exc)


@app.route('/api/decipher', methods=['POST'])
def decipher():
    data  = request.get_json(silent=True)
    if not data:
        return _api_error('MISSING_INPUT', 'JSON body required')
    codes = data.get('codes', [])
    if not codes:
        return _api_error('MISSING_INPUT', '"codes" list is required')
    if not isinstance(codes, list) or not all(isinstance(c, str) for c in codes):
        return _api_error('INVALID_INPUT', '"codes" must be a list of strings')

    gardiner_str     = ' '.join(codes).upper()
    result, err      = _run_pipeline_safe(gardiner_str)
    if err:
        return _api_error('PIPELINE_ERROR', err)

    code_list  = gardiner_str.split()
    tla_tokens = result['tla_phonemes'].split()
    meanings   = result['meanings']
    per_sign   = []
    for i, c in enumerate(code_list):
        key  = c.strip().lower()
        info = GARDINER_MAP.get(key, {})
        per_sign.append([
            c, info.get('unicode', ''),
            tla_tokens[i] if i < len(tla_tokens) else '',
            meanings[i]   if i < len(meanings)   else '',
        ])
    glyphs_str = ''.join(
        GARDINER_MAP.get(c.strip().lower(), {}).get('unicode', '')
        for c in code_list
    )
    intent = result.get('intention', {})
    sent   = result.get('sentiment', {})
    return jsonify({
        'success': True,
        'data': {
            'path'             : result['path'],
            'glyphs'           : glyphs_str,
            'per_sign'         : per_sign,
            'tla_norm'         : result['tla_normalised'],
            'english'          : result['english'],
            'arabic'           : result['arabic'],
            'arabic_verified'  : result.get('arabic_verified', None),
            'arabic_confidence': result.get('arabic_confidence', None),
            'sentiment'        : sent.get('label', 'neutral').lower(),
            'sent_score'       : f"{sent.get('confidence', 0):.0%}",
            'intention_en'     : intent.get('intention_en', ''),
            'intention_ar'     : intent.get('intention_ar', ''),
            'intent_confidence': intent.get('confidence', None),
            'rag_score'        : result['rag_score'],
            'time'             : result['processing_time'],
        }
    }), 200


@app.route('/api/translate', methods=['POST'])
def api_translate():
    data     = request.get_json(silent=True)
    if not data:
        return _api_error('MISSING_INPUT', 'JSON body required')
    gardiner = data.get('gardiner', '').strip()
    if not gardiner:
        return _api_error('MISSING_INPUT', '"gardiner" field is required')

    result, err = _run_pipeline_safe(gardiner)
    if err:
        return _api_error('PIPELINE_ERROR', err)

    return jsonify({
        'success'        : True,
        'input'          : result['input'],
        'path'           : result['path'],
        'tla_phonemes'   : result['tla_phonemes'],
        'tla_normalised' : result['tla_normalised'],
        'meanings'       : result['meanings'],
        'english'        : result['english'],
        'arabic'         : result['arabic'],
        'arabic_verified': result.get('arabic_verified'),
        'sentiment'      : result.get('sentiment'),
        'intention'      : result.get('intention'),
        'rag_score'      : result['rag_score'],
        'rag_results'    : result.get('rag_results', [])[:3],
        'time'           : result['processing_time'],
    }), 200


@app.route('/api/evaluate', methods=['POST'])
def api_evaluate():
    data     = request.get_json(silent=True)
    if not data:
        return _api_error('MISSING_INPUT', 'JSON body required')
    gardiner = data.get('gardiner', '').strip()
    if not gardiner:
        return _api_error('MISSING_INPUT', '"gardiner" field is required')
    ref_en   = data.get('reference_english', '')
    ref_de   = data.get('reference_german',  '')

    try:
        result = evaluate_pipeline(gardiner, reference_english=ref_en,
                                   reference_german=ref_de)
    except Exception as e:
        return _api_error('PIPELINE_ERROR', str(e))

    eval_metrics = {k: result[k] for k in [
        'bleu','rouge1','rouge2','rougeL','meteor','chrf',
        'exact_match','word_overlap',
        'recall@1','recall@3','recall@5','recall@10','recall@20',
        'mrr','avg_retrieval_score',
    ] if k in result}
    return jsonify({'success': True, 'input': gardiner,
                    'english': result['english'], 'metrics': eval_metrics}), 200


@app.route('/api/status', methods=['GET'])
def status():
    return jsonify({
        'status'        : 'ready',
        'version'       : 'v8-Graduation-Final',
        'environment'   : ENV,
        'model'         : 'Seed-X-PPO-7B',
        'signs_loaded'  : len(GARDINER_MAP),
        'faiss_vectors' : faiss_index.ntotal if faiss_index else 0,
        'test_set_rows' : len(df_test_global) if df_test_global is not None else 0,
        'intention_classes': len(INTENTION_MAP),
        'arabic_verification': True,
        'prompt_token_limit': MAX_PROMPT_TOKENS,
    }), 200

@app.route('/api/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'version': 'v8-Graduation-Final'}), 200

@app.route('/api/examples', methods=['GET'])
def examples():
    return jsonify({
        'simple_word'   : {'codes': ['O1'],                'description': 'House'},
        'monument'      : {'codes': ['G17', 'N35', 'D21'], 'description': 'Monument'},
        'sun_disk'      : {'codes': ['N5'],                'description': 'Sun / Ra'},
        'son_of_ra'     : {'codes': ['O4', 'N5'],          'description': 'Son of Ra'},
        'royal_offering': {'codes': ['R4', 'X8', 'A42'],   'description': 'Offering of the king'},
    }), 200


def run_server():
    app.run(host='0.0.0.0', port=5010, use_reloader=False, debug=False)

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)

tunnel     = ngrok.connect(5010)
PUBLIC_URL = tunnel.public_url

print('=' * 65)
print('🏛️  HIEROGLYPH NLP PIPELINE v8 — GRADUATION FINAL')
print(f'🌐  Public URL:\n\n    {PUBLIC_URL}\n')
print('📡  Endpoints:')
print(f'      POST {PUBLIC_URL}/api/decipher')
print(f'      POST {PUBLIC_URL}/api/translate')
print(f'      POST {PUBLIC_URL}/api/evaluate')
print(f'      GET  {PUBLIC_URL}/api/status')
print(f'      GET  {PUBLIC_URL}/api/health')
print(f'      GET  {PUBLIC_URL}/api/examples')
print('=' * 65)


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5010
 * Running on http://172.19.2.2:5010
Press CTRL+C to quit


🏛️  HIEROGLYPH NLP PIPELINE v8 — GRADUATION FINAL
🌐  Public URL:

    https://irretrievably-unsimpering-darrin.ngrok-free.dev

📡  Endpoints:
      POST https://irretrievably-unsimpering-darrin.ngrok-free.dev/api/decipher
      POST https://irretrievably-unsimpering-darrin.ngrok-free.dev/api/translate
      POST https://irretrievably-unsimpering-darrin.ngrok-free.dev/api/evaluate
      GET  https://irretrievably-unsimpering-darrin.ngrok-free.dev/api/status
      GET  https://irretrievably-unsimpering-darrin.ngrok-free.dev/api/health
      GET  https://irretrievably-unsimpering-darrin.ngrok-free.dev/api/examples


127.0.0.1 - - [05/Apr/2026 04:30:35] "OPTIONS /api/examples HTTP/1.1" 200 -
127.0.0.1 - - [05/Apr/2026 04:30:36] "GET /api/examples HTTP/1.1" 200 -
127.0.0.1 - - [05/Apr/2026 04:30:39] "OPTIONS /api/decipher HTTP/1.1" 200 -
127.0.0.1 - - [05/Apr/2026 04:31:02] "POST /api/decipher HTTP/1.1" 200 -
127.0.0.1 - - [05/Apr/2026 04:31:13] "OPTIONS /api/decipher HTTP/1.1" 200 -
